## Projet Santé Mentale des Adolescents

In [187]:
# Dépendances du notebook
%pip install openpyxl==3.1.3 pandas==3.0.2 s3fs==2026.3.0 -q

Note: you may need to restart the kernel to use updated packages.


## Importation des packages nécessaires

In [188]:
import pandas as pd
import os
import openpyxl
from openpyxl import *
from openpyxl.worksheet.formula import ArrayFormula
from openpyxl.utils import *
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from openpyxl import Workbook   
from openpyxl.utils.dataframe import dataframe_to_rows
from openpyxl import load_workbook 
from openpyxl.utils import get_column_letter
from openpyxl.styles import PatternFill
from openpyxl.chart import BarChart, Reference
from openpyxl.styles import Font, Border, Side
from openpyxl.styles import Alignment
from openpyxl.chart.label import DataLabelList                                                                                                                                                      
from openpyxl.worksheet.datavalidation import DataValidation
from openpyxl.worksheet.formula import ArrayFormula
from openpyxl.utils import quote_sheetname
from openpyxl.utils.cell import coordinate_from_string, column_index_from_string
from openpyxl.worksheet.worksheet import Worksheet
from openpyxl.styles import Alignment, PatternFill
from openpyxl.worksheet.datavalidation import DataValidation
import pandas as pd
from PIL import Image

print(openpyxl.__version__)


3.1.3


### Importation des données - Santé Mentale

Après l'importation, on inspecte les types de données présents.

In [189]:
df = pd.read_csv('https://minio.lab.sspcloud.fr/nerojeni10/DATA_PROJET_SMA/Teen_Mental_Health_Dataset.csv')

df.info()
df.head()

<class 'pandas.DataFrame'>
RangeIndex: 1200 entries, 0 to 1199
Data columns (total 13 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   age                       1200 non-null   int64  
 1   gender                    1200 non-null   str    
 2   daily_social_media_hours  1200 non-null   float64
 3   platform_usage            1200 non-null   str    
 4   sleep_hours               1200 non-null   float64
 5   screen_time_before_sleep  1200 non-null   float64
 6   academic_performance      1200 non-null   float64
 7   physical_activity         1200 non-null   float64
 8   social_interaction_level  1200 non-null   str    
 9   stress_level              1200 non-null   int64  
 10  anxiety_level             1200 non-null   int64  
 11  addiction_level           1200 non-null   int64  
 12  depression_label          1200 non-null   int64  
dtypes: float64(5), int64(5), str(3)
memory usage: 140.4 KB


,age,gender,daily_social_media_hours,platform_usage,sleep_hours,screen_time_before_sleep,academic_performance,physical_activity,social_interaction_level,stress_level,anxiety_level,addiction_level,depression_label
0,14,male,7.9,Instagram,7.4,2.9,3.01,1.5,low,2,2,1,0
1,19,female,1.9,TikTok,8.0,2.9,3.22,0.8,high,8,1,10,0
2,17,female,1.3,Instagram,7.6,0.5,3.92,0.0,high,2,4,2,0
3,15,male,7.4,TikTok,6.9,1.6,3.48,0.8,medium,1,7,9,0
4,15,female,4.7,Both,4.9,3.0,2.37,1.4,medium,3,5,2,0


### Inspecter la présence des valeurs manquantes

In [190]:
df.isnull().sum()

age                         0
gender                      0
daily_social_media_hours    0
platform_usage              0
sleep_hours                 0
screen_time_before_sleep    0
academic_performance        0
physical_activity           0
social_interaction_level    0
stress_level                0
anxiety_level               0
addiction_level             0
depression_label            0
dtype: int64

***Il n'y a aucune valeur manquante dans ce jeux de données***

## Remplissage du fichier

Pour remplir le fichier, on allons créer plusieurs feuilles composées des données nécessaires à la création des indicateurs

In [191]:
path_file = "../template/Projet_ODD_SIVARAJAH.xlsx"

# Recréer un fichier propre sans feuille parasite
try:
    wb = load_workbook(path_file)  # noqa: F405
except Exception:
    wb = Workbook()


# Créer un vrai fichier Excel vide si inexistant
if not os.path.exists(path_file):
    wb = Workbook()
    wb.save(path_file)


# Ajouter la feuille DATA
with pd.ExcelWriter(path_file, mode="a", if_sheet_exists="replace") as writer:
    df.to_excel(writer, sheet_name='DATA', index=False)



print("Feuilles présentes :", load_workbook(path_file).sheetnames)

Feuilles présentes : ['Sheet', 'DATA']


## Création de la feuille CALC

Sur cette feuille, on  aura les valeurs distinctes pour chaque variable, afin de réaliser des groupes plus tard et de réaliser des agrégations dessus.

In [192]:
# # Chargement du fichier en mémoire
# wb = load_workbook(path_file)

# # Créer la feuille CALC si elle n'existe pas
# if "CALC" not in wb.sheetnames:
#     ws = wb.create_sheet("CALC")
# else:
#     ws = wb["CALC"]

### Création des variables distinctes

In [193]:
# from openpyxl.utils import FORMULAE
# "UNIQUE" in FORMULAE

# # ws["A1"]="Genres distincts"
# formula = "=_xlfn.UNIQUE(DATA!B2:B)"
# ws["A1"]=ArrayFormula("A1:A", formula)

# # # ws["A3"]="Ages distincts"
# # # ws["A3"]= "=_xlfn.UNIQUE(DATA!A2:A)"

# # # ws["A5"]="Plateformes distincts"
# # # ws["A5"]= "=_xlfn.UNIQUE(DATA!D2:D)"

# # # ws["A7"]="Social Interactions"
# # # ws["A7"]= "=_xlfn.UNIQUE(DATA!I2:I)"


# # wb.save(path_file)


## Création des indicateurs

In [194]:
# Chargement du fichier en mémoire
wb = load_workbook(path_file)


# Supprimer la feuille vide par défaut si elle existe
if "Sheet" in wb.sheetnames:
    del wb["Sheet"]

wb.save(path_file)

# Créer la feuille Indicateurs si elle n'existe pas
if "Indicateurs" not in wb.sheetnames:
    ws = wb.create_sheet("Indicateurs")
else:
    ws = wb["Indicateurs"]

# Ajout des formules
# 1. Nombre de filles dépressives
ws['A1'] = "Nombre de filles dépressives"
ws['B1'] = '=COUNTIFS(DATA!M:M,1,DATA!B:B,"female")'

# 2. Nombre de garçons dépressifs
ws['A2'] = "Nombre de garçons dépressifs"
ws['B2'] = '=COUNTIFS(DATA!M:M,1,DATA!B:B,"male")'

# 3. Niveau d'addiction moyen chez les filles
ws['A3'] = "Niveau d'addiction moyen chez les filles"
ws['B3'] = '=AVERAGEIF(DATA!B:B,"female",DATA!L:L)'

# 4. Niveau d'addiction moyen chez les garçons
ws['A4'] = "Niveau d'addiction moyen chez les garçons"
ws['B4'] = '=AVERAGEIF(DATA!B:B,"male",DATA!L:L)'


# Création d'une nouvelle feuille, TCD (Tableau croisé dynamique)

Sur cette feuille apparaîtrant les indicateurs qui sont groupés selon différents critères comme l'âge ou le genre. 

Pour plus de simplicité, la réalisation de ces groupes et des agrégations nécessaires j'utilise la bibliothèque pandas et les résultats sont par la suite transcris dans les feuilles. 

Afin d'automatiser l'écriture des données, et d'éviter le chevauchement des résultats une fonction est crée pour CALCuler automatiquement la cellule dans la  quelle on commencera à écrire les données.

In [195]:
def write_table(ws, df, start_row, title=None, space=3, padding=2):
    """
    Écrit un DataFrame dans une feuille Excel OpenPyXL à partir d'une ligne donnée.

    La fonction ajoute éventuellement un titre, écrit les en-têtes de colonnes
    puis les données du DataFrame. Elle retourne ensuite la première ligne
    disponible pour écrire un nouveau tableau en laissant un nombre de lignes
    vides configurable.

    Args:
        ws: Feuille OpenPyXL cible.
        df: DataFrame à écrire.
        start_row (int): Ligne de départ.
        title (str, optional): Titre du tableau.
        space (int, optional): Nombre de lignes vides à laisser après le tableau.

    Returns:
        int: Numéro de la prochaine ligne disponible.

    Examples:
    >>> start_row = 1
    >>> start_row = write_table(ws, tcd1, start_row,
    ...                         "Dépression selon l'âge")
    >>> start_row = write_table(ws, tcd2, start_row,
    ...                         "Addiction moyenne selon l'âge et le genre")
    """

    # En-têtes
    if title:
        ws.cell(row=start_row, column=1, value=title)
        start_row += 1

    # En-têtes + ajustement largeur colonnes
    for col_idx, header in enumerate(df.columns, start=1):
        ws.cell(row=start_row, column=col_idx, value=header)

        col_letter = get_column_letter(col_idx)
        width = len(str(header)) + padding

        # On conserve la plus grande largeur si la colonne existe déjà
        current_width = ws.column_dimensions[col_letter].width
        if current_width is None or width > current_width:
            ws.column_dimensions[col_letter].width = width

    # Données
    for i, row in df.iterrows():
        for col_idx, value in enumerate(row, start=1):
            ws.cell(
                row=start_row + i + 1,
                column=col_idx,
                value=value
            )

    # Ligne de départ du tableau suivant
    return start_row + len(df) + space + 1

### Création de la feuille TCD si elle n'existe pas

In [196]:
if "TCD" not in wb.sheetnames:
    ws_tcd = wb.create_sheet("TCD")
else:
    ws_tcd = wb["TCD"]


### Création des indicateurs et écriture des données avec la fonction créée

In [197]:
# 5. Niveau de dépression selon l'âge
tcd1 = df.groupby(["age"])["depression_label"].sum().reset_index()
tcd1.columns = ["Age", "Nb depressifs"]

start_row = 1
start_row = write_table(
    ws_tcd,
    tcd1,
    start_row,
    title="Niveau de dépression selon l'âge"
)
# 6. Niveau d'addiction moyen selon l'âge et le genre
tcd2 = df.groupby(["age", "gender"])["addiction_level"].mean().round(2).unstack()
tcd2.columns.name = None
tcd2 = tcd2.reset_index()
tcd2.columns = ["Age", "Addiction moy. Filles", "Addiction moy. Garcons"]

start_row = write_table(
    ws_tcd,
    tcd2,
    start_row,
    title="Niveau d'addiction moyen selon l'âge et le genre"
)

# 7. Nombre de depressions selon le temps de sommeil et l'interaction sociale
df["sleep_group"] = pd.cut(df["sleep_hours"],
                            bins=[0, 5, 6, 7, 8, 12],
                            labels=["<5h", "5-6h", "6-7h", "7-8h", ">8h"])
tcd3 = df.groupby(["social_interaction_level", "sleep_group"])["depression_label"].sum().unstack()
tcd3.columns.name = None
tcd3 = tcd3.reset_index()
tcd3.columns = ["Interaction sociale/Temps de Sommeil", "<5h", "5-6h", "6-7h", "7-8h", ">8h"]

start_row = write_table(
    ws_tcd,
    tcd3,
    start_row,
    title="Nombre de dépressions selon le temps de sommeil et l'interaction sociale"
)

# 8. Repartition niveau addiction selon l'age (indicateurs boite a moustaches)
summary_box = df.groupby("age")["addiction_level"].agg(
    Q1      = lambda x: x.quantile(0.25),
    Mediane = lambda x: x.quantile(0.50),
    Q3      = lambda x: x.quantile(0.75),
    Min     = "min",
    Max     = "max",
    Moyenne = "mean",
    IQR     = lambda x: x.quantile(0.75) - x.quantile(0.25)
).round(2).reset_index()
summary_box.columns = ["Age", "Q1", "Mediane", "Q3", "Min", "Max", "Moyenne", "IQR"]

start_row = write_table(
    ws_tcd,
    summary_box,
    start_row,
    title="Répartition du niveau d'addiction selon l'âge"
)

# Sauvegarde du fichier
wb.save(path_file)

### Création d'une matrice de corrélation

Pour étudier les liens entre les différents variables de ce jeux de données

In [198]:
# Encoder les variables catégorielles en numérique
df["gender_num"] = df["gender"].map({"male": 0, "female": 1})
df["social_num"] = df["social_interaction_level"].map({"low": 0, "medium": 1, "high": 2})

# Sélectionner les colonnes numériques
cols_corr = ["age", "sleep_hours", "daily_social_media_hours",
             "academic_performance", "physical_activity",
             "social_num", "stress_level", "anxiety_level",
             "addiction_level", "depression_label"]

# Matrice de corrélation
corr = df[cols_corr].corr().round(2)

corr_reset = corr.reset_index()
corr_reset.columns = ["Variable"] + cols_corr

# Création d'une nouvelle feuille pour réaliser le tableau de corrélation
wb = load_workbook(path_file)

if "Correlations" not in wb.sheetnames:
    ws_corr = wb.create_sheet("Correlations")
else:
    ws_corr = wb["Correlations"]

write_table(
    ws_corr,
    corr_reset,
    start_row=1,
    title="Matrice de corrélation",
    space=0
)

# Sauvegarde du fichier
wb.save(path_file)
wb.close()
print("Feuilles presentes :", load_workbook(path_file).sheetnames)

Feuilles presentes : ['DATA', 'Indicateurs', 'TCD', 'Correlations']


In [199]:

cols_to_calculate = ['age', 'gender', 'platform_usage', 'social_interaction_level']
len_dict ={}
for col in cols_to_calculate:
    len_dict[f"len_{col}"] = len(df[col].unique())+1 
print(f'{len_dict}')



{'len_age': 8, 'len_gender': 3, 'len_platform_usage': 4, 'len_social_interaction_level': 4}


In [200]:
from openpyxl import load_workbook
from openpyxl.worksheet.formula import ArrayFormula
from openpyxl.worksheet.table import Table, TableStyleInfo

wb = load_workbook(path_file)

if "CALC" not in wb.sheetnames:
    ws_calc = wb.create_sheet("CALC")
else:
    ws_calc = wb["CALC"]

style = TableStyleInfo(
    name="TableStyleMedium2",
    showFirstColumn=False,
    showLastColumn=False,
    showRowStripes=True,
    showColumnStripes=False
)

# Genres - colonne A
formula = "=_xlfn.UNIQUE(DATA!B:B)"
ws_calc['A1'] = ArrayFormula(
    f"A1:A{len_dict['len_gender']}",
    formula
)
table_gender = Table(displayName="tblGenres", ref=f"A1:A{len_dict['len_gender']}")
table_gender.tableStyleInfo = style
table_gender.hasHeader = False
ws_calc.add_table(table_gender)

# Ages - colonne C
formula = "=_xlfn.UNIQUE(DATA!A:A)"
ws_calc['C1'] = ArrayFormula(
    f"C1:C{len_dict['len_age']}",
    formula
)
table_age = Table(displayName="tblAges", ref=f"C1:C{len_dict['len_age']}")
table_age.tableStyleInfo = style
table_age.hasHeader = False
ws_calc.add_table(table_age)

# Plateformes - colonne E
formula = "=_xlfn.UNIQUE(DATA!D:D)"
ws_calc['E1'] = ArrayFormula(
    f"E1:E{len_dict['len_platform_usage']}",
    formula
)
table_platform = Table(displayName="tblPlateformes", ref=f"E1:E{len_dict['len_platform_usage']}")
table_platform.tableStyleInfo = style
table_platform.hasHeader = False
ws_calc.add_table(table_platform)

# Interactions sociales - colonne G
formula = "=_xlfn.UNIQUE(DATA!I:I)"
ws_calc['G1'] = ArrayFormula(
    f"G1:G{len_dict['len_social_interaction_level']}",
    formula
)
table_interaction = Table(displayName="tblInteractions", ref=f"G1:G{len_dict['len_social_interaction_level']}")
table_interaction.tableStyleInfo = style
table_interaction.hasHeader = False
ws_calc.add_table(table_interaction)

wb.save(path_file)
wb.close()
print("Feuilles presentes :", load_workbook(path_file).sheetnames)

Feuilles presentes : ['DATA', 'Indicateurs', 'TCD', 'Correlations', 'CALC']


/opt/python/lib/python3.13/site-packages/openpyxl/worksheet/_writer.py:274: UserWarning: File may not be readable: column headings must be strings.
  warn("File may not be readable: column headings must be strings.")


# Création du dashboard

## Création des filtres

Maintenant qu'on a les indicateurs uniques, on peut les utiliser pour la création de filtres.

In [201]:
from openpyxl import load_workbook
from openpyxl.styles import Alignment, PatternFill
from openpyxl.worksheet.datavalidation import DataValidation

wb = load_workbook(path_file)

# Création de la première page du tableau de bord
if "TDB_1" not in wb.sheetnames:
    TDB1_sheet = wb.create_sheet("TDB_1")
else:
    TDB1_sheet = wb["TDB_1"]

# Retirer la grille 
TDB1_sheet.sheet_view.showGridLines = False 

# Titre du filtre
filter_1_cell = TDB1_sheet['A1']
filter_1_cell.value = 'Genre'
filter_1_cell.alignment = Alignment(horizontal='center', vertical='center')
filter_1_cell.fill = PatternFill(start_color='00C0C0C0', end_color='00C0C0C0', fill_type='solid')

# Fusionner les cellules A1:B2
TDB1_sheet.merge_cells('A1:B2')

# Valeur du filtre 
val_filter_1_cell = TDB1_sheet['C1']
val_filter_1_cell.value = "female"
val_filter_1_cell.alignment = Alignment(horizontal='center', vertical='center')
val_filter_1_cell.fill = PatternFill(start_color='00C0C0C0', end_color='00C0C0C0', fill_type='solid')

# Créer une liste de validation de données avec les valeurs de la colonne A de CALC
formula = f"=CALC!$A$2:$A${len_dict['len_gender']}"

# Ajouter la liste de validation pour créer le filtre
dv = DataValidation(type='list', formula1=formula)
TDB1_sheet.add_data_validation(dv)
dv.add('C1')

wb.save(path_file)
wb.close()

In [202]:
def add_filter(worksheet, title_col, title_row, title_text, value_col, value_row, 
               data_source_col, len_data, default_value=None, color='00C0C0C0'):
    """
    Ajoute un filtre (titre + liste de validation) à un tableau de bord Excel.
    
    Parameters
    ----------
    worksheet : openpyxl.worksheet.worksheet.Worksheet
        La feuille Excel où ajouter le filtre
    title_col : str
        Colonne du titre (ex: 'A')
    title_row : int
        Ligne du titre (ex: 1)
    title_text : str
        Texte du titre du filtre (ex: 'Genre')
    value_col : str
        Colonne de la valeur/sélection (ex: 'C')
    value_row : int
        Ligne de la valeur/sélection (ex: 1)
    data_source_col : str
        Colonne source en CALC (ex: 'A')
    len_data : int
        Nombre de valeurs uniques dans la source
    default_value : str, optional
        Valeur par défaut du filtre (ex: 'female')
    color : str, optional
        Couleur de fond en hexadécimal (default: '00C0C0C0' = gris clair)
    """
    
    fill = PatternFill(start_color=color, end_color=color, fill_type='solid')
    alignment = Alignment(horizontal='center', vertical='center')
    border = Border(
        left=Side(style='thin'),
        right=Side(style='thin'),
        top=Side(style='thin'),
        bottom=Side(style='thin')
    )
    
    # Cellule du titre
    title_cell = worksheet[f'{title_col}{title_row}']
    title_cell.value = title_text
    title_cell.alignment = alignment
    title_cell.fill = fill
    title_cell.border = border
    title_cell.font = Font(bold=True)
    
    # Fusionner les cellules pour le titre (2x2)
    end_col = chr(ord(title_col) + 1)
    end_row = title_row + 1
    worksheet.merge_cells(f'{title_col}{title_row}:{end_col}{end_row}')
    
    # Cellule de la valeur du filtre
    value_cell = worksheet[f'{value_col}{value_row}']
    value_cell.value = default_value
    value_cell.alignment = alignment
    value_cell.fill = fill
    value_cell.border = border
    
    # Formule de validation (référence à CALC)
    # Format : =CALC!$A$2:$A$5  (de la ligne 2 à len_data+1)
    formula = f"=CALC!${data_source_col}$2:${data_source_col}${len_data+1}"
    
    # Ajouter la liste de validation
    dv = DataValidation(type='list', formula1=formula)
    worksheet.add_data_validation(dv)
    dv.add(f'{value_col}{value_row}')
    
    print(f"✅ Filtre '{title_text}' créé (Formule: {formula})")
 
# ============================================================================
# 4. CRÉER/RÉCUPÉRER LA FEUILLE TDB_1
# ============================================================================
 
print("\nCréation des filtres pour TDB_1...")
 
if "TDB_1" not in wb.sheetnames:
    TDB1_sheet = wb.create_sheet("TDB_1")
else:
    TDB1_sheet = wb["TDB_1"]
 
# Retirer la grille
TDB1_sheet.sheet_view.showGridLines = False
 
# ============================================================================
# 5. AJOUTER LES FILTRES À TDB_1
# ============================================================================
 
# Utilisation avec valeurs par défaut
add_filter(TDB1_sheet, 'A', 1, 'Genre', 'C', 1, 'A', len_dict['len_gender'], default_value='female')
add_filter(TDB1_sheet, 'A', 4, 'Âge', 'C', 4, 'C', len_dict['len_age'], default_value='18')
add_filter(TDB1_sheet, 'A', 7, 'Plateforme', 'C', 7, 'E', len_dict['len_platform_usage'], default_value='Instagram')
add_filter(TDB1_sheet, 'A', 10, 'Interaction', 'C', 10, 'G', len_dict['len_social_interaction_level'], default_value='High')
 
# ============================================================================
# 6. AJOUTER LES FILTRES À TDB_2 (Optionnel)
# ============================================================================
 
print("\n Création des filtres pour TDB_2...")
 
if "TDB_2" not in wb.sheetnames:
    TDB2_sheet = wb.create_sheet("TDB_2")
else:
    TDB2_sheet = wb["TDB_2"]
 
TDB2_sheet.sheet_view.showGridLines = False
 
# Ajouter les mêmes filtres
add_filter(TDB2_sheet, 'A', 1, 'Genre', 'C', 1, 'A', len_dict['len_gender'], default_value='female')
add_filter(TDB2_sheet, 'A', 4, 'Âge', 'C', 4, 'C', len_dict['len_age'], default_value='18')
 
# ============================================================================
# 7. SAUVEGARDER (Une seule fois à la fin !)
# ============================================================================
 
print("\n Sauvegarde du workbook...")
wb.save(path_file)
wb.close()


Création des filtres pour TDB_1...
✅ Filtre 'Genre' créé (Formule: =CALC!$A$2:$A$4)
✅ Filtre 'Âge' créé (Formule: =CALC!$C$2:$C$9)
✅ Filtre 'Plateforme' créé (Formule: =CALC!$E$2:$E$5)
✅ Filtre 'Interaction' créé (Formule: =CALC!$G$2:$G$5)

 Création des filtres pour TDB_2...
✅ Filtre 'Genre' créé (Formule: =CALC!$A$2:$A$4)
✅ Filtre 'Âge' créé (Formule: =CALC!$C$2:$C$9)

 Sauvegarde du workbook...


# Création des tableaux groupés

Ces tableaux permettront de créer les graphiques reposant sur plusieurs critères par exemple.

In [203]:
from openpyxl import load_workbook
from openpyxl.utils import get_column_letter
from openpyxl.styles import PatternFill, Font, Alignment, Border, Side

wb = load_workbook(path_file)

header_fill = PatternFill(start_color="4472C4", end_color="4472C4", fill_type="solid")
header_font = Font(color="FFFFFF", bold=True)
row_header_fill = PatternFill(start_color="E7E6E6", end_color="E7E6E6", fill_type="solid")
row_header_font = Font(bold=True)
title_fill = PatternFill(start_color="70AD47", end_color="70AD47", fill_type="solid")
title_font = Font(bold=True, size=12, color="FFFFFF")
border = Border(
    left=Side(style='thin'),
    right=Side(style='thin'),
    top=Side(style='thin'),
    bottom=Side(style='thin')
)
center_align = Alignment(horizontal="center", vertical="center")

# 1. Créer la feuille
if "TAB_CROISE_DYNAMIQUE" in wb.sheetnames:
    del wb["TAB_CROISE_DYNAMIQUE"]
ws_tcd = wb.create_sheet("TAB_CROISE_DYNAMIQUE")


# Répartition de la dépression selon le genre
ws_tcd['A1'] = "Dépression par Âge et Genre"
ws_tcd['A1'].font = title_font
ws_tcd['A1'].fill = title_fill
ws_tcd.merge_cells('A1:H1')

ws_tcd['A2'] = "Genre"
ws_tcd['A2'].fill = header_fill
ws_tcd['A2'].font = header_font
ws_tcd['A2'].border = border

# 5. EN-TÊTES COLONNES (Âges de CALC)
for col_idx in range(1, 8):
    col_letter = get_column_letter(col_idx + 1)  # B, C, D, E, F, G, H
    
    # Récupérer l'âge unique #col_idx depuis CALC!C:C
    ws_tcd[f'{col_letter}2'] = f'=IFERROR(INDEX(CALC!$C$2:$C$1000,{col_idx}),"")'
    ws_tcd[f'{col_letter}2'].fill = header_fill
    ws_tcd[f'{col_letter}2'].font = header_font
    ws_tcd[f'{col_letter}2'].border = border

# 6. EN-TÊTES LIGNES (Genres de CALC)
for row_idx in range(1, 3):
    row_num = 2 + row_idx  # 3, 4, 5, 6
    
    # Récupérer le genre unique #row_idx depuis CALC!A:A
    ws_tcd[f'A{row_num}'] = f'=IFERROR(INDEX(CALC!$A$2:$A$1000,{row_idx}),"")'
    ws_tcd[f'A{row_num}'].fill = row_header_fill
    ws_tcd[f'A{row_num}'].font = row_header_font
    ws_tcd[f'A{row_num}'].border = border

# 7. REMPLIR LES CELLULES DE DONNÉES (Croiser Genre × Âge)
for row_idx in range(1, 3):
    row_num = 2 + row_idx
    
    for col_idx in range(1, 8):
        col_letter = get_column_letter(col_idx + 1)
        
        # CORRECTION : Utilisation de COUNTIFS (Anglais) avec la bonne syntaxe de comptage
        formula = f'=IFERROR(COUNTIFS(DATA!$M:$M,1,DATA!$A:$A,${col_letter}$2,DATA!$B:$B,$A{row_num}),0)'
        
        ws_tcd[f'{col_letter}{row_num}'] = formula
        ws_tcd[f'{col_letter}{row_num}'].border = border
        ws_tcd[f'{col_letter}{row_num}'].number_format = '0'

# 8. Ajuster les largeurs
ws_tcd.column_dimensions['A'].width = 18
for col_idx in range(1, 8):
    ws_tcd.column_dimensions[get_column_letter(col_idx + 1)].width = 12

# 9. Sauvegarder
wb.save(path_file)
wb.close()

print("TCD créé dans la feuille 'TAB_CROISE_DYNAMIQUE'")

TCD créé dans la feuille 'TAB_CROISE_DYNAMIQUE'


In [204]:
# =================================================================
# NOUVEAU TABLEAU : Addiction moyenne par Âge et Genre
# =================================================================

start_row = 10 # On commence à la ligne 10 pour ne pas écraser le 1er tableau

ws_tcd[f'A{start_row}'] = "Addiction moyenne par Âge et Genre"
ws_tcd[f'A{start_row}'].font = title_font
ws_tcd[f'A{start_row}'].fill = title_fill
ws_tcd.merge_cells(f'A{start_row}:H{start_row}')

ws_tcd[f'A{start_row+1}'] = "Genre"
ws_tcd[f'A{start_row+1}'].fill = header_fill
ws_tcd[f'A{start_row+1}'].font = header_font
ws_tcd[f'A{start_row+1}'].border = border

# EN-TÊTES COLONNES (Âges)
for col_idx in range(1, 8):
    col_letter = get_column_letter(col_idx + 1)
    
    ws_tcd[f'{col_letter}{start_row+1}'] = f'=IFERROR(INDEX(CALC!$C$2:$C$1000,{col_idx}),"")'
    ws_tcd[f'{col_letter}{start_row+1}'].fill = header_fill
    ws_tcd[f'{col_letter}{start_row+1}'].font = header_font
    ws_tcd[f'{col_letter}{start_row+1}'].border = border

# EN-TÊTES LIGNES (Genres)
# Toujours range(1, 3) puisqu'il y a 2 genres
for row_idx in range(1, 3):
    row_num = start_row + 1 + row_idx 
    
    ws_tcd[f'A{row_num}'] = f'=IFERROR(INDEX(CALC!$A$2:$A$1000,{row_idx}),"")'
    ws_tcd[f'A{row_num}'].fill = row_header_fill
    ws_tcd[f'A{row_num}'].font = row_header_font
    ws_tcd[f'A{row_num}'].border = border

# REMPLIR LES CELLULES (Croiser Genre × Âge pour la Moyenne)
for row_idx in range(1, 3):
    row_num = start_row + 1 + row_idx
    
    for col_idx in range(1, 8):
        col_letter = get_column_letter(col_idx + 1)
        
        # AVERAGEIFS = Moyenne si conditions multiples
        # Plage à moyenner : DATA!$L:$L (Niveau d'addiction)
        formula = f'=IFERROR(AVERAGEIFS(DATA!$L:$L,DATA!$A:$A,${col_letter}${start_row+1},DATA!$B:$B,$A{row_num}),0)'
        
        ws_tcd[f'{col_letter}{row_num}'] = formula
        ws_tcd[f'{col_letter}{row_num}'].border = border
        # Affichage avec 2 chiffres après la virgule pour plus de lisibilité
        ws_tcd[f'{col_letter}{row_num}'].number_format = '0.00' 

# Sauvegarder à nouveau
wb.save(path_file)

In [205]:
from openpyxl.worksheet.formula import ArrayFormula
from openpyxl.utils import get_column_letter

# Summary pour boite à moustache
start_row = 16  # Ligne de départ pour laisser de l'espace avec le tableau du dessus

# 1. TITRE DU TABLEAU
ws_tcd[f'A{start_row}'] = "Répartition de l'addiction selon l'âge"
ws_tcd[f'A{start_row}'].font = title_font
ws_tcd[f'A{start_row}'].fill = title_fill
ws_tcd.merge_cells(f'A{start_row}:G{start_row}')

# 2. EN-TÊTES DES COLONNES
colonnes = ["Âge", "Min", "Q1", "Médiane", "Q3", "Max", "Moyenne"]
for col_idx, col_name in enumerate(colonnes):
    col_letter = get_column_letter(col_idx + 1)
    ws_tcd[f'{col_letter}{start_row+1}'] = col_name
    ws_tcd[f'{col_letter}{start_row+1}'].fill = header_fill
    ws_tcd[f'{col_letter}{start_row+1}'].font = header_font
    ws_tcd[f'{col_letter}{start_row+1}'].border = border
    ws_tcd[f'{col_letter}{start_row+1}'].alignment = center_align

# Nombre d'âges uniques (détermine le nombre de lignes à générer automatiquement)
num_ages = len(df['age'].unique())

# 3. REMPLISSAGE DU TABLEAU AVEC LES FORMULES EXCEL
for row_idx in range(1, num_ages + 1):
    row_num = start_row + 1 + row_idx
    
    # Colonne A : Récupération de l'âge depuis la feuille CALC
    ws_tcd[f'A{row_num}'] = f'=IFERROR(INDEX(CALC!$C$2:$C$1000,{row_idx}),"")'
    ws_tcd[f'A{row_num}'].fill = row_header_fill
    ws_tcd[f'A{row_num}'].font = row_header_font
    ws_tcd[f'A{row_num}'].border = border
    ws_tcd[f'A{row_num}'].alignment = center_align
    
    # Colonne B : Minimum conditionnel
    ws_tcd[f'B{row_num}'] = ArrayFormula(
        f'B{row_num}', 
        f'=MIN(IF(DATA!$A$2:$A$1201=$A{row_num}, DATA!$L$2:$L$1201))'
    )
    
    # Colonne C : Premier Quartile (Formule Matricielle)
    ws_tcd[f'C{row_num}'] = ArrayFormula(
        f'C{row_num}', 
        f'=QUARTILE(IF(DATA!$A$2:$A$1201=$A{row_num}, DATA!$L$2:$L$1201), 1)'
    )
    
    # Colonne D : Médiane (Formule Matricielle)
    ws_tcd[f'D{row_num}'] = ArrayFormula(
        f'D{row_num}', 
        f'=MEDIAN(IF(DATA!$A$2:$A$1201=$A{row_num}, DATA!$L$2:$L$1201))'
    )
    
    # Colonne E : Troisième Quartile (Formule Matricielle)
    ws_tcd[f'E{row_num}'] = ArrayFormula(
        f'E{row_num}', 
        f'=QUARTILE(IF(DATA!$A$2:$A$1201=$A{row_num}, DATA!$L$2:$L$1201), 3)'
    )
    
    # Colonne F : Maximum conditionnel
   # Colonne F : Maximum conditionnel
    ws_tcd[f'F{row_num}'] = ArrayFormula(
        f'F{row_num}', 
        f'=MAX(IF(DATA!$A$2:$A$1201=$A{row_num}, DATA!$L$2:$L$1201))'
    )
    
    # Colonne G : Moyenne conditionnelle
    ws_tcd[f'G{row_num}'] = f'=AVERAGEIFS(DATA!$L$2:$L$1201, DATA!$A$2:$A$1201, $A{row_num})'
    
    # 4. APPLICATION DES STYLES POUR LES COLONNES DE DONNÉES (B à G)
    for col_idx in range(2, 8):
        col_letter = get_column_letter(col_idx)
        cell = ws_tcd[f'{col_letter}{row_num}']
        cell.border = border
        cell.alignment = center_align
        cell.number_format = '0.00'  # Format propre avec deux décimales

# Réseau le plus utilisé 

ws_tcd["K1"] = "Réseau le plus utilisé"
cellule_cible = 'L1' 

formule_plateforme = '=INDEX(DATA!$D$2:$D$1201, MATCH(MAX(COUNTIFS(DATA!$D$2:$D$1201, DATA!$D$2:$D$1201, DATA!$D$2:$D$1201, "<>Both")), COUNTIFS(DATA!$D$2:$D$1201, DATA!$D$2:$D$1201, DATA!$D$2:$D$1201, "<>Both"), 0))'

ws_tcd[cellule_cible] = ArrayFormula(cellule_cible, formule_plateforme)

# Titre de la cellule
ws_tcd["K2"] = "Perf sco moyenne - dépressifs"

# La formule en anglais avec les virgules
formule_dep = "=AVERAGEIF(DATA!M:M, 1, DATA!G:G)"
ws_tcd["L2"] = formule_dep

# Sauvegarde finale
wb.save(path_file)
print("Tableau 100% formules Excel créé avec succès !")

Tableau 100% formules Excel créé avec succès !


In [206]:
# Définissez les lignes de départ pour espacer vos tableaux
start_row_t1 = 30  # Tableau 1 : Nb dépressions par âge
start_row_t2 = 45  # Tableau 2 : Perf scolaire par genre et temps d'écran
start_row_t3 = 55  # Tableau 3 : Dépression par interaction et temps de sommeil

# =================================================================
# TABLEAU 1 : Nombre de dépressions selon l'âge
# =================================================================

ws_tcd[f'A{start_row_t1}'] = "Nombre de dépressions selon l'âge"
ws_tcd[f'A{start_row_t1}'].font = title_font
ws_tcd[f'A{start_row_t1}'].fill = title_fill
ws_tcd.merge_cells(f'A{start_row_t1}:B{start_row_t1}')

ws_tcd[f'A{start_row_t1+1}'] = "Âge"
ws_tcd[f'B{start_row_t1+1}'] = "Nb Dépressions"
for col in ['A', 'B']:
    ws_tcd[f'{col}{start_row_t1+1}'].fill = header_fill
    ws_tcd[f'{col}{start_row_t1+1}'].font = header_font
    ws_tcd[f'{col}{start_row_t1+1}'].border = border
    ws_tcd[f'{col}{start_row_t1+1}'].alignment = center_align

num_ages = len(df['age'].unique()) # Déjà calculé, sert pour la boucle

for row_idx in range(1, num_ages + 1):
    row_num = start_row_t1 + 1 + row_idx
    
    # Colonne A : Âge
    ws_tcd[f'A{row_num}'] = f'=IFERROR(INDEX(CALC!$C$2:$C$1000,{row_idx}),"")'
    ws_tcd[f'A{row_num}'].fill = row_header_fill
    ws_tcd[f'A{row_num}'].font = row_header_font
    ws_tcd[f'A{row_num}'].border = border
    ws_tcd[f'A{row_num}'].alignment = center_align
    
    # Colonne B : Countifs (Age = Age Ligne, Depression = 1)
    ws_tcd[f'B{row_num}'] = f'=COUNTIFS(DATA!$A$2:$A$1201, $A{row_num}, DATA!$M$2:$M$1201, 1)'
    ws_tcd[f'B{row_num}'].border = border
    ws_tcd[f'B{row_num}'].alignment = center_align


# =================================================================
# TABLEAU 2 : Perf. scolaire selon Genre et Temps d'écran
# =================================================================

ws_tcd[f'A{start_row_t2}'] = "Performance scolaire selon Genre et Temps d'écran (Réseaux Sociaux)"
ws_tcd[f'A{start_row_t2}'].font = title_font
ws_tcd[f'A{start_row_t2}'].fill = title_fill
ws_tcd.merge_cells(f'A{start_row_t2}:E{start_row_t2}')

colonnes_t2 = ["Genre", "< 2h", "2 à 4h", "4 à 6h", "> 6h"]
for col_idx, col_name in enumerate(colonnes_t2):
    col_letter = get_column_letter(col_idx + 1)
    ws_tcd[f'{col_letter}{start_row_t2+1}'] = col_name
    ws_tcd[f'{col_letter}{start_row_t2+1}'].fill = header_fill
    ws_tcd[f'{col_letter}{start_row_t2+1}'].font = header_font
    ws_tcd[f'{col_letter}{start_row_t2+1}'].border = border
    ws_tcd[f'{col_letter}{start_row_t2+1}'].alignment = center_align

# Il n'y a que 2 genres (Male, Female)
for row_idx in range(1, 3):
    row_num = start_row_t2 + 1 + row_idx
    
    # Colonne A : Genre depuis CALC (colonne A)
    ws_tcd[f'A{row_num}'] = f'=IFERROR(INDEX(CALC!$A$2:$A$1000,{row_idx}),"")'
    ws_tcd[f'A{row_num}'].fill = row_header_fill
    ws_tcd[f'A{row_num}'].font = row_header_font
    ws_tcd[f'A{row_num}'].border = border
    
    # Colonnes B à E : AVERAGEIFS (Moyenne si...) avec tranches horaires (Colonne C de DATA)
    # < 2h
    ws_tcd[f'B{row_num}'] = f'=IFERROR(AVERAGEIFS(DATA!$G$2:$G$1201, DATA!$B$2:$B$1201, $A{row_num}, DATA!$C$2:$C$1201, "<2"), 0)'
    # 2 à 4h
    ws_tcd[f'C{row_num}'] = f'=IFERROR(AVERAGEIFS(DATA!$G$2:$G$1201, DATA!$B$2:$B$1201, $A{row_num}, DATA!$C$2:$C$1201, ">=2", DATA!$C$2:$C$1201, "<4"), 0)'
    # 4 à 6h
    ws_tcd[f'D{row_num}'] = f'=IFERROR(AVERAGEIFS(DATA!$G$2:$G$1201, DATA!$B$2:$B$1201, $A{row_num}, DATA!$C$2:$C$1201, ">=4", DATA!$C$2:$C$1201, "<6"), 0)'
    # > 6h
    ws_tcd[f'E{row_num}'] = f'=IFERROR(AVERAGEIFS(DATA!$G$2:$G$1201, DATA!$B$2:$B$1201, $A{row_num}, DATA!$C$2:$C$1201, ">=6"), 0)'
    
    for col_letter in ['B', 'C', 'D', 'E']:
        ws_tcd[f'{col_letter}{row_num}'].border = border
        ws_tcd[f'{col_letter}{row_num}'].alignment = center_align
        ws_tcd[f'{col_letter}{row_num}'].number_format = '0.00'


# =================================================================
# TABLEAU 3 : Anxiété moyenne selon Temps de sommeil et Interaction
# =================================================================

# Mise à jour du titre
ws_tcd[f'A{start_row_t3}'] = "Anxiété moyenne par Interaction sociale et Temps de sommeil"
ws_tcd[f'A{start_row_t3}'].font = title_font
ws_tcd[f'A{start_row_t3}'].fill = title_fill
ws_tcd.merge_cells(f'A{start_row_t3}:F{start_row_t3}')

# En-têtes des colonnes (inchangés)
colonnes_t3 = ["Interaction", "< 5h", "5 à 6h", "6 à 7h", "7 à 8h", "> 8h"]
for col_idx, col_name in enumerate(colonnes_t3):
    col_letter = get_column_letter(col_idx + 1)
    ws_tcd[f'{col_letter}{start_row_t3+1}'] = col_name
    ws_tcd[f'{col_letter}{start_row_t3+1}'].fill = header_fill
    ws_tcd[f'{col_letter}{start_row_t3+1}'].font = header_font
    ws_tcd[f'{col_letter}{start_row_t3+1}'].border = border
    ws_tcd[f'{col_letter}{start_row_t3+1}'].alignment = center_align

# Remplissage des données (3 niveaux d'interaction)
for row_idx in range(1, 4):
    row_num = start_row_t3 + 1 + row_idx
    
    # Colonne A : Niveau d'interaction
    ws_tcd[f'A{row_num}'] = f'=IFERROR(INDEX(CALC!$G$2:$G$1000,{row_idx}),"")'
    ws_tcd[f'A{row_num}'].fill = row_header_fill
    ws_tcd[f'A{row_num}'].font = row_header_font
    ws_tcd[f'A{row_num}'].border = border
    
    # Colonnes B à F : AVERAGEIFS (Moyenne colonne K si interaction=X, sommeil=tranche)
    # J'ajoute IFERROR(..., 0) pour éviter les #DIV/0! si une case est vide
    
    # < 5h
    ws_tcd[f'B{row_num}'] = f'=IFERROR(AVERAGEIFS(DATA!$K$2:$K$1201, DATA!$I$2:$I$1201, $A{row_num}, DATA!$E$2:$E$1201, "<5"), 0)'
    # 5 à 6h
    ws_tcd[f'C{row_num}'] = f'=IFERROR(AVERAGEIFS(DATA!$K$2:$K$1201, DATA!$I$2:$I$1201, $A{row_num}, DATA!$E$2:$E$1201, ">=5", DATA!$E$2:$E$1201, "<6"), 0)'
    # 6 à 7h
    ws_tcd[f'D{row_num}'] = f'=IFERROR(AVERAGEIFS(DATA!$K$2:$K$1201, DATA!$I$2:$I$1201, $A{row_num}, DATA!$E$2:$E$1201, ">=6", DATA!$E$2:$E$1201, "<7"), 0)'
    # 7 à 8h
    ws_tcd[f'E{row_num}'] = f'=IFERROR(AVERAGEIFS(DATA!$K$2:$K$1201, DATA!$I$2:$I$1201, $A{row_num}, DATA!$E$2:$E$1201, ">=7", DATA!$E$2:$E$1201, "<8"), 0)'
    # > 8h
    ws_tcd[f'F{row_num}'] = f'=IFERROR(AVERAGEIFS(DATA!$K$2:$K$1201, DATA!$I$2:$I$1201, $A{row_num}, DATA!$E$2:$E$1201, ">=8"), 0)'
    
    # Application des bordures et du format numérique (2 décimales pour une moyenne)
    for col_letter in ['B', 'C', 'D', 'E', 'F']:
        ws_tcd[f'{col_letter}{row_num}'].border = border
        ws_tcd[f'{col_letter}{row_num}'].alignment = center_align
        ws_tcd[f'{col_letter}{row_num}'].number_format = '0.00'

# Sauvegarde finale
wb.save(path_file)
print("Les trois tableaux ont été créés avec succès !")


Les trois tableaux ont été créés avec succès !
